# Multi-period Planning for Petroleum Procurement $\rightarrow$ Pooling $\rightarrow$ Product Blending

The "procurement planning department" of a mid-sized petroleum refining and procurement company decides over a planning horizon of several weeks which raw materials (sweet/sour crude oil, various distillates) to buy, when, how much, which intermediate tanks (pools) to put them in, and which products (premium/regular, etc.) to blend them into from there.

Tanks are treated as "well-stirred mixtures," and their sulfur concentration is determined by **sulfur mass inventory / volume inventory**. Since sulfur flows out at this concentration upon discharge, **bilinear terms of concentration $\times$ flow rate and concentration $\times$ inventory** inherently appear — this is not an approximation, but the strong non-convex structure itself classically known as the pooling problem.

Business meaning of each constraint:

- **Procurement contract on/off (fixed cost + minimum take amount)**: A fixed cost is incurred when contracting with a supplier for a period, and for contracted periods, there is an obligation to buy at least the minimum take amount (take-or-pay style).
- **Tank inventory carrying over periods**: Purchased raw materials can be stored in intermediate tanks and discharged in later periods (time coupling).
- **Product quality specifications**: Each product has a specification upper limit for sulfur content, and the weighted average concentration of the blend must not exceed it.
- **Spot market backstop**: Any shortfall that cannot be covered by in-house blending is filled by relatively expensive spot purchases (a realistic escape hatch that always ensures feasibility).

Subject: `samples/manufacturing_and_blending/petroleum_pooling.py`. In this notebook, we follow what makes this problem difficult as a single story, in the order of "Naive formulation $\rightarrow$ Diagnosis $\rightarrow$ Inside the diagnosis $\rightarrow$ Trying improvements".

In [ ]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/manufacturing_and_blending"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import petroleum_pooling as pp
from pyscipopt import Model, quicksum

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

Load `build_model(scale="default")` and check the scale and location of non-linear terms. The default scale is 8 raw materials $\times$ 5 pools $\times$ 4 products $\times$ 8 periods.

In [ ]:
m0 = pp.build_model("default")
nl = sum(1 for c in m0.getConss() if c.isNonlinear())
print(f"変数 {m0.getNVars()}(うち整数/バイナリ {m0.getNBinVars()})、制約 {m0.getNConss()}"
      f"(うち非線形 {nl})")
print("非線形制約の例(先頭5件、名前で種類がわかる):")
for c in m0.getConss():
    if c.isNonlinear():
        print(" -", c.name)
del m0

`conc_def_*` (definition of concentration: `smass == conc*inv`) and `sulfur_balance_*` (sulfur mass balance: containing terms like `conc*outflow`) appear for the number of pools $\times$ periods. Both are bilinearities of **concentration (continuous) $\times$ volume/flow rate (continuous)**, arising from the physics itself that the contents of the pool are a "well-stirred mixture."

## 2. Diagnosis

Actually solve with `mk.analyze` to collect stagnation of dual bounds, violations of non-linear constraints, and coefficient scales.

In [ ]:
report = mk.analyze(lambda: pp.build_model("default"), name="petroleum-pooling",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.1%}",
      "/ 支配ボトルネック:", report.metrics.get("bottleneck_type"),
      f"(相対違反 {report.metrics.get('bottleneck_rel_viol', 0):.2f})")

In 30 seconds, 1 instance of `numerical_scale` (order of magnitude difference in coefficient scales remaining after presolve / Big-M candidates) triggers.
The gap is not very large at about 3.8% — although the relative violation of `conc_def` (concentration definition) is large ($\approx 1.0$, producing concentrations that are almost meaningless in the root LP relaxation solution), `spatial_share` (dual contribution of spatial branching) is low, and the `weak_relaxation` rule (which triggers when spatial branching is dominant) does not meet its conditions — suggesting that SCIP's branching process is able to absorb a considerable amount of the non-linear weakness on the discrete branching side. Next, we will directly examine the contents of this `numerical_scale` (what is being picked up as Big-M candidates).

## 3. Looking Inside the Diagnosis — Static Diagnosis (Coefficient Scale)

Directly hit the `mk.static_diag` family of functions to check the large coefficients (Big-M candidates) that remain after presolve.

In [ ]:
from minlpkit.collectors.static_diag import extract_coefficients, scale_summary, residual_scale

m1 = pp.build_model("default")
rs = residual_scale(m1)
print(f"presolve後の係数比 = {rs['ratio']:.3g}(中央値 {rs['median']:.3g})")
bigm_df = None
import pandas as pd
bigm_df = pd.DataFrame(rs["bigm"])
bigm_df

In [ ]:
fig = go.Figure(layout=base_layout(
    "presolve後に残る大係数(Big-M候補、中央値比で上位10件)",
    "制約/変数名", "係数の絶対値", height=420))
fig.add_trace(go.Bar(x=bigm_df["name"], y=bigm_df["magnitude"],
    marker_color=[C["warn"] if s == "制約係数" else C["s1"] for s in bigm_df["source"]],
    text=bigm_df["source"], textposition="outside"))
fig.update_xaxes(tickangle=-30)
show(fig)

At the top are constraints around the take-or-pay link of procurement contracts (`contract_ub_*`/`contract_lb_*`), which take the form of **Big-M type linked constraints** like `quicksum(x) <= avail[i,t]*z[i,t]`. The M here is `avail[i,t]` (the maximum amount that can actually be supplied in that period), which is not an arbitrarily large constant but a physical upper limit itself, but it is still larger as a "constraint coefficient" compared to other terms (like inventory storage cost coefficients, which are on the order of less than 1), catching the threshold on the `residual_bigm_count` side (100 times the median).

## 4. Trying Improvements — Replacing Big-M Type Linked Constraints with Indicator Constraints

`contract_ub`/`contract_lb` expresses the logical constraint "purchase is 0 if not contracted (`z=0`), and respects upper limit avail/lower limit min_buy if contracted (`z=1`)" using Big-M. SCIP's `addConsIndicator` passes this logic directly to the solver, eliminating the need to choose a value for M (the same technique as [Method Guide 3. Eliminating Big-M](../../playbook/03-bigm.md)). Since `petroleum_pooling.py` itself has no argument to switch variants, we assemble the Indicator version within this notebook using the same data generation (`pp._data`) as `build_model` (reusing the `z`/`x` variables of the existing model as they are, and replacing just the 2 Big-M constraints with 3 Indicator constraints).

In [ ]:
def build_indicator(scale="default"):
    '''contract_ub/contract_lb (Big-M) を addConsIndicator に置き換えた変種。'''
    m = pp.build_model(scale)
    z, x = m.data["z"], m.data["x"]
    d = pp._data(scale)
    nF, nL, nT = d["nF"], d["nL"], d["nT"]
    avail, min_buy = d["avail"], d["min_buy"]
    for c in list(m.getConss()):
        if c.name.startswith("contract_ub_") or c.name.startswith("contract_lb_"):
            m.delCons(c)
    for i in range(nF):
        for t in range(nT):
            tot = quicksum(x[i, l, t] for l in range(nL))
            # z=0 なら購入0(Big-Mを使わない厳密な論理制約)
            m.addConsIndicator(tot <= 0, binvar=z[i, t], activeone=False,
                               name=f"ind_off_{i}_{t}")
            # z=1 のときだけ上限avail・下限min_buyを課す
            m.addConsIndicator(tot <= float(avail[i, t]), binvar=z[i, t], activeone=True,
                               name=f"ind_cap_{i}_{t}")
            m.addConsIndicator(-tot <= -float(min_buy[i]), binvar=z[i, t], activeone=True,
                               name=f"ind_lb_{i}_{t}")
    return m

# 等価性の最小確認(small scaleで同じ最適値に到達するか)
mb = pp.build_model("small"); mb.hideOutput(); mb.optimize()
mi = build_indicator("small"); mi.hideOutput(); mi.optimize()
print(f"baseline (Big-M) : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"indicator        : obj={mi.getObjVal():.2f}  status={mi.getStatus()}")

It reaches the same optimal value at the small scale (different representations of equivalent logical constraints). Next, at the default scale, we compare **root dual bound, final gap, and node count** using `mk.compare_variants`.

In [ ]:
df = mk.compare_variants(
    {"baseline (Big-M)": lambda: pp.build_model("default"),
     "indicator":        lambda: build_indicator("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
base = df.loc[df["variant"] == "baseline (Big-M)"].iloc[0]
ind  = df.loc[df["variant"] == "indicator"].iloc[0]
labels = ["baseline (Big-M)", "indicator"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], ind["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, ind["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], ind["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="Big-M -> Indicator の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {ind['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.1%} -> {ind['final_gap']:.1%}")
print(f"nodes     : {int(base['nodes'])} -> {int(ind['nodes'])}")

**Honest verification result**: The Big-M (`avail[i,t]`) in this model is inherently a **tight value without arbitrariness** representing "the maximum amount that can actually be bought in that period," and lacks the "useless looseness" of the loose Big-M in notebook 03 (fixed_charge). Therefore, the improvement in `root_dual`/`final_gap`/`nodes` by Indicatorization tends to be limited (at almost the same level). Even so, the diagnosis picked up `numerical_scale` because it looks at the "**order of magnitude difference** from other coefficients (like storage costs)" rather than the validity of the value itself — in other words, this finding is not a "model defect" but a **natural numerical range** reflecting the "practical reality that the scales of procurement costs and storage costs are fundamentally different," and Indicatorization does not touch the main cause of the gap (bilinearity of sulfur concentration).
If we truly want to affect the gap, the approach (McCormick strengthening, variable bound tightening) to the `conc_def`/`sulfur_balance` side (concentration $\times$ flow rate bilinearity) seen in Section 2 will be the main route.

## Summary

- The essential difficulty of this problem lies in the bilinearity (pooling problem) of concentration $\times$ flow rate and concentration $\times$ inventory, born from **the realistic operation of tanks "storing, mixing, and discharging later"**. The Big-M pointed to by `numerical_scale` in the diagnosis is actually a naturally business-appropriate numerical range of "the difference in scale between procurement costs and storage costs," and does not dramatically improve even with Indicatorization — **the learning of this notebook is the honest confirmation that the symptom picked up by the diagnosis and the true cause dominating the gap are on different axes**.
- Practically, this decision is an integrated plan of procurement, inventory, and blending: "which raw materials to buy when, which tanks to store them in, and when to discharge them into which products." The strength of the bilinearity comes not from an approximation but from the physics and chemistry (mixing rules) themselves. SCIP handles this strictly with a spatial branch-and-bound method, but as the scale increases, the weakness of the root relaxation (concentration $\times$ flow rate) becomes more pronounced.

Related: [Method Guide 3. Eliminating Big-M](../../playbook/03-bigm.md) /
[Method Guide 8. Condition Number and Numerical Soundness](../../playbook/08-condition.md) /
API [`mk.analyze`](../../api/pipeline.md)